# UKHSA chart regression: NHS calls to GP admissions

This notebook uses UKHSA dashboard export files already present in the repository. It looks for files whose names start with `ukhsa-chart`, identifies NHS-call series and GP/admission series, then fits lagged regressions.

The first pass is intentionally transparent: inspect the inferred file catalogue before fitting the model, and override the file choices if the automatic classification is wrong.

## Model

The baseline model is:

```text
z(GP admissions)_t = beta_0 + beta_1 z(NHS calls)_t + beta_2 z(NHS calls)_{t-1} + ... + seasonality + error_t
```

Use weekly frequency first if the UKHSA files are weekly; switch to monthly if needed.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from wastewater.io import find_repo_root
from wastewater.ukhsa import (
    find_ukhsa_chart_files,
    build_ukhsa_series_catalogue,
    chart_to_series,
    build_regression_frame_from_ukhsa,
    fit_ukhsa_lagged_ols,
)

ROOT = find_repo_root(ROOT)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
ROOT

## 1. Find UKHSA chart files

This scans the whole checkout for files beginning with `ukhsa-chart`.

In [ ]:
ukhsa_files = find_ukhsa_chart_files(ROOT)
display(ukhsa_files.sort_values('path'))

print(f'Found {len(ukhsa_files)} UKHSA chart files')

## 2. Infer file roles and schemas

Check the `series_type`, `date_column`, and `value_column` columns. If a file has been misclassified, override it in the next cell.

In [ ]:
catalogue = build_ukhsa_series_catalogue(ROOT)
cols = ['path', 'filename', 'series_type', 'date_column', 'value_column', 'size_kb']
display(catalogue[cols].sort_values(['series_type', 'path']))

# Full column lists, useful for debugging source schemas.
display(catalogue[['path', 'columns']])

## 3. Choose predictor and outcome files

The automatic defaults use files classified as:

- `nhs_calls` for the predictor
- `gp_admissions` for the outcome

If that is wrong, set `PREDICTOR_FILES` and `OUTCOME_FILES` manually using paths from the catalogue above.

In [ ]:
PREDICTOR_FILES = catalogue.loc[catalogue['series_type'] == 'nhs_calls', 'path'].tolist()
OUTCOME_FILES = catalogue.loc[catalogue['series_type'] == 'gp_admissions', 'path'].tolist()

# Manual override example:
# PREDICTOR_FILES = ['data/raw/ukhsa-chart-...nhs111...csv']
# OUTCOME_FILES = ['data/raw/ukhsa-chart-...gp-admissions...csv']

print('Predictor files:')
for path in PREDICTOR_FILES:
    print(' -', path)

print('\nOutcome files:')
for path in OUTCOME_FILES:
    print(' -', path)

## 4. Preview selected series

In [ ]:
for path in PREDICTOR_FILES[:3]:
    print('\nNHS-call predictor:', path)
    display(chart_to_series(ROOT, path, series_type='nhs_calls').head())

for path in OUTCOME_FILES[:3]:
    print('\nGP/admission outcome:', path)
    display(chart_to_series(ROOT, path, series_type='gp_admissions').head())

## 5. Build regression frame

Use weekly aggregation first. Change `FREQ = 'M'` if the UKHSA chart files are monthly.

In [ ]:
FREQ = 'W'
LAGS = [0, 1, 2, 3, 4]

regression_frame = build_regression_frame_from_ukhsa(
    ROOT,
    predictor_files=PREDICTOR_FILES,
    outcome_files=OUTCOME_FILES,
    freq=FREQ,
    lags=LAGS,
)

display(regression_frame)

out = PROCESSED / 'ukhsa_nhs_calls_gp_admissions_regression_frame.csv'
regression_frame.to_csv(out, index=False)
print(f'Saved {len(regression_frame):,} rows to {out.relative_to(ROOT)}')

## 6. Plot the aligned series

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(regression_frame['period'], regression_frame['z_nhs_calls'], label='NHS calls, z-scored')
ax.plot(regression_frame['period'], regression_frame['z_gp_admissions'], label='GP admissions, z-scored')
ax.set_title('UKHSA NHS calls and GP admissions')
ax.set_xlabel('Period')
ax.set_ylabel('z-score')
ax.legend()
fig.autofmt_xdate()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(regression_frame['z_nhs_calls'], regression_frame['z_gp_admissions'])
ax.set_xlabel('NHS calls, z-scored')
ax.set_ylabel('GP admissions, z-scored')
ax.set_title('Contemporaneous association')
plt.show()

## 7. Fit lagged regression

The result uses HAC standard errors to reduce sensitivity to autocorrelation in the residuals.

In [ ]:
result = fit_ukhsa_lagged_ols(regression_frame, lags=LAGS, seasonal_controls=True)
print(result.summary())

## 8. Observed versus fitted

In [ ]:
predictor_cols = [f'z_nhs_calls_lag{lag}' for lag in LAGS]
model_rows = regression_frame.dropna(subset=['z_gp_admissions', *predictor_cols]).copy()
model_rows['prediction'] = result.predict()
model_rows['residual'] = model_rows['z_gp_admissions'] - model_rows['prediction']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(model_rows['period'], model_rows['z_gp_admissions'], label='Observed GP admissions, z')
ax.plot(model_rows['period'], model_rows['prediction'], label='Predicted GP admissions, z')
ax.set_title('Observed versus fitted GP admissions')
ax.legend()
fig.autofmt_xdate()
plt.show()

model_rows.to_csv(PROCESSED / 'ukhsa_nhs_calls_gp_admissions_model_rows.csv', index=False)
model_rows[['period', 'z_gp_admissions', 'prediction', 'residual']].head()

## 9. Interpretation checklist

- Check whether `lag0`, `lag1`, etc. are clinically plausible.
- Compare the model with and without seasonal controls.
- Inspect residual autocorrelation.
- Try weekly versus monthly aggregation if the source date scale is ambiguous.
- Add wastewater respiratory pathogen predictors only after the NHS-call baseline is stable.